Import CollegeBasketballData and other needed requirements. 

In [41]:
import pandas as pd
import requests
import notebook
import sqlalchemy
import psycopg2
import json
import numpy as np
from datetime import datetime
import cbbd
import glob 
import os 
from datetime import datetime, timedelta

 

Pull key froms json

In [4]:
with open("keys.json", "r") as f:
    keys = json.load(f)
    API_KEY = keys.get("API_KEY")


  

Import CSV data to database. 


In [31]:
files = glob.glob("data/NCAA Statistics *.csv")

dfs = []
for f in files:
    temp = pd.read_csv(f)
    temp["year"] = os.path.basename(f).replace("NCAA Statistics ", "").replace(".csv", "")
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

# Save to a new CSV
df = df[["year"] + [col for col in df.columns if col != "year"]]
df.to_csv("data/NCAA Statistics Combined.csv", index=False)

print(f"Done! Combined {len(files)} files with {len(df)} total rows.")
print(df.head())

Done! Combined 8 files with 5024 total rows.
       year           Team     Conference  NET Rank  PrevNET  AvgOppNETRank  \
0  Combined        Gonzaga            WCC         1        1            108   
1  Combined         Kansas         Big 12         2        2              1   
2  Combined         Dayton    Atlantic 10         3        3             89   
3  Combined  San Diego St.  Mountain West         4        4            110   
4  Combined         Baylor         Big 12         5        5             46   

   AvgOppNET    WL Conf.Record Non-ConferenceRecord RoadWL    SOS  NonConfSOS  \
0        146  31-2        17-1                 14-1   10-1  111.0       286.0   
1         67  27-3        17-1                 11-2   10-1    1.0         1.0   
2        130  29-2        18-0                 11-2    9-0   27.0        48.0   
3        150  29-2        19-2                 11-0   11-0   98.0       145.0   
4         97  26-4        15-3                 11-1    9-2   58.0       169

Calc WAB for seasons in range

In [60]:
seasons = range(2006, 2027)
bubble_percentile = 0.15
all_results = []

configuration = cbbd.Configuration(access_token=API_KEY)

def win_prob_vs_bubble(opponent_elo: float, bubble_elo: float) -> float:
    diff = opponent_elo - bubble_elo
    prob = 1.0 / (1.0 + 10 ** (diff / 400))
    return float(np.clip(prob, 0.05, 0.95))

for season in seasons:
    print(f"\nFetching season {season}...")
    year1 = season - 1
    year2 = season
    feb_end = datetime(year2, 3, 1) - timedelta(days=1)

    with cbbd.ApiClient(configuration) as api_client:
        games_raw_1 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year1, 11, 1),
            end_date_range=datetime(year1, 12, 31),
            status=cbbd.GameStatus.FINAL
        )
        games_raw_2 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year2, 1, 1),
            end_date_range=feb_end,
            status=cbbd.GameStatus.FINAL
        )
        games_raw_3 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year2, 3, 1),
            end_date_range=datetime(year2, 3, 14),
            status=cbbd.GameStatus.FINAL
        )
        elo_raw = cbbd.RatingsApi(api_client).get_elo(season=season)

    games_raw = games_raw_1 + games_raw_2 + games_raw_3

    games_list = []
    for game in games_raw:
        games_list.append({
            'homeTeam': game.home_team,
            'awayTeam': game.away_team,
            'homeScore': game.home_points,
            'awayScore': game.away_points,
        })

    games = pd.DataFrame(games_list).drop_duplicates().reset_index(drop=True)
    games["homeScore"] = pd.to_numeric(games["homeScore"], errors="coerce")
    games["awayScore"] = pd.to_numeric(games["awayScore"], errors="coerce")

    elo = pd.DataFrame([rating.to_dict() for rating in elo_raw])
    elo_sorted = elo.sort_values("elo", ascending=False).reset_index(drop=True)
    bubble_index = min(int(len(elo_sorted) * bubble_percentile), len(elo_sorted) - 1)
    bubble_elo = elo_sorted.iloc[bubble_index]["elo"]

    d1_teams = set(elo_sorted["team"])
    games = games[
        (games["homeTeam"].isin(d1_teams)) &
        (games["awayTeam"].isin(d1_teams))
    ].reset_index(drop=True)

    print(f"Season : {season}")
    print(f"Teams  : {len(elo_sorted)}")
    print(f"Games  : {len(games)}")
    print(f"Bubble ELO threshold (top {int(bubble_percentile*100)}%): {bubble_elo:.1f}")

    elo_dict = dict(zip(elo_sorted["team"], elo_sorted["elo"]))
    rank_lookup = {row["team"]: i + 1 for i, row in elo_sorted.iterrows()}

    all_teams = set(games["homeTeam"]) | set(games["awayTeam"])
    results = []

    for team in all_teams:
        team_games = games[
            (games["homeTeam"] == team) | (games["awayTeam"] == team)
        ]

        actual_wins = 0
        expected_wins = 0.0
        games_played = 0

        for _, game in team_games.iterrows():
            if game["homeTeam"] == team:
                opponent  = game["awayTeam"]
                my_score  = game["homeScore"]
                opp_score = game["awayScore"]
            else:
                opponent  = game["homeTeam"]
                my_score  = game["awayScore"]
                opp_score = game["homeScore"]

            if pd.isna(my_score) or pd.isna(opp_score):
                continue

            if my_score > opp_score:
                actual_wins += 1

            opp_elo = elo_dict.get(opponent, 1500)
            expected_wins += win_prob_vs_bubble(opp_elo, bubble_elo)
            games_played += 1

        if games_played == 0:
            continue

        results.append({
            "season":        season,
            "team":          team,
            "wab":           round(actual_wins - expected_wins, 3),
            "actual_wins":   actual_wins,
            "expected_wins": round(expected_wins, 3),
            "games_played":  games_played,
            "elo":           round(elo_dict.get(team, 1500), 1),
            "elo_rank":      rank_lookup.get(team),
        })

    all_results.extend(results)
    print(f"Season {season} done — {len(results)} teams calculated")

# Final combined DataFrame — outside the season loop
WAB_df = (
    pd.DataFrame(all_results)
    .sort_values(["season", "wab"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nTop 25 per season:\n")
for season in seasons:
    print(f"\n--- {season} ---")
    season_df = WAB_df[WAB_df["season"] == season].copy()
    season_df.insert(0, "rank", range(1, len(season_df) + 1))
    print(season_df.to_string(index=False))


Fetching season 2006...
Season : 2006
Teams  : 326
Games  : 4449
Bubble ELO threshold (top 15%): 1838.0
Season 2006 done — 326 teams calculated

Fetching season 2007...
Season : 2007
Teams  : 326
Games  : 4701
Bubble ELO threshold (top 15%): 1880.0
Season 2007 done — 326 teams calculated

Fetching season 2008...
Season : 2008
Teams  : 328
Games  : 4664
Bubble ELO threshold (top 15%): 1872.0
Season 2008 done — 328 teams calculated

Fetching season 2009...
Season : 2009
Teams  : 330
Games  : 4678
Bubble ELO threshold (top 15%): 1853.0
Season 2009 done — 330 teams calculated

Fetching season 2010...
Season : 2010
Teams  : 335
Games  : 4824
Bubble ELO threshold (top 15%): 1888.0
Season 2010 done — 335 teams calculated

Fetching season 2011...
Season : 2011
Teams  : 346
Games  : 5177
Bubble ELO threshold (top 15%): 1868.0
Season 2011 done — 346 teams calculated

Fetching season 2012...
Season : 2012
Teams  : 350
Games  : 5152
Bubble ELO threshold (top 15%): 1853.0
Season 2012 done — 345 te

Combined NCAA stats

In [42]:
NCAA_Data = pd.read_csv("data/NCAA Statistics Combined.csv")

In [45]:
print(NCAA_Data.shape)
print(NCAA_Data.dtypes)
NCAA_Data.head()

(5024, 17)
year                        str
Team                        str
Conference                  str
NET Rank                  int64
PrevNET                   int64
AvgOppNETRank             int64
AvgOppNET                 int64
WL                          str
Conf.Record                 str
Non-ConferenceRecord        str
RoadWL                      str
SOS                     float64
NonConfSOS              float64
Q1                          str
Q2                          str
Q3                          str
Q4                          str
dtype: object


,year,Team,Conference,NET Rank,PrevNET,AvgOppNETRank,AvgOppNET,WL,Conf.Record,Non-ConferenceRecord,RoadWL,SOS,NonConfSOS,Q1,Q2,Q3,Q4
0,Combined,Gonzaga,WCC,1,1,108,146,31-2,17-1,14-1,10-1,111.0,286.0,6-2,5-0,10-0,10-0
1,Combined,Kansas,Big 12,2,2,1,67,27-3,17-1,11-2,10-1,1.0,1.0,12-3,8-0,4-0,3-0
2,Combined,Dayton,Atlantic 10,3,3,89,130,29-2,18-0,11-2,9-0,27.0,48.0,5-2,8-0,9-0,7-0
3,Combined,San Diego St.,Mountain West,4,4,110,150,29-2,19-2,11-0,11-0,98.0,145.0,4-1,7-0,8-1,10-0
4,Combined,Baylor,Big 12,5,5,46,97,26-4,15-3,11-1,9-2,58.0,169.0,11-2,5-2,6-0,4-0


In [46]:
for columns in NCAA_Data:
    print(NCAA_Data.isnull().sum())

year                      0
Team                      0
Conference                0
NET Rank                  0
PrevNET                   0
AvgOppNETRank             0
AvgOppNET                 0
WL                        0
Conf.Record               0
Non-ConferenceRecord      0
RoadWL                    0
SOS                     726
NonConfSOS              758
Q1                        0
Q2                        0
Q3                        0
Q4                        0
dtype: int64
year                      0
Team                      0
Conference                0
NET Rank                  0
PrevNET                   0
AvgOppNETRank             0
AvgOppNET                 0
WL                        0
Conf.Record               0
Non-ConferenceRecord      0
RoadWL                    0
SOS                     726
NonConfSOS              758
Q1                        0
Q2                        0
Q3                        0
Q4                        0
dtype: int64
year                  

In [47]:
print(f"There are {NCAA_Data.duplicated().sum()} duplicated rows.")

print(NCAA_Data['Team'].value_counts())

print(NCAA_Data['Q1'].value_counts())

print(NCAA_Data['Q2'].value_counts())

There are 0 duplicated rows.
Team
Kansas              14
Dayton              14
Baylor              14
Michigan St.        14
Louisville          14
                    ..
LIU(AQ)              2
UMBC(AQ)             2
Lehigh(AQ)           2
Prairie View(AQ)     2
New Haven            2
Name: count, Length: 464, dtype: int64
Q1
0-2     840
0-1     814
0-3     622
0-0     346
0-4     292
       ... 
10-9      2
9-11      2
8-10      2
3-2       2
3-0       2
Name: count, Length: 170, dtype: int64
Q2
0-1     490
0-2     478
0-3     454
0-0     302
0-4     272
       ... 
9-5       2
7-6       2
2-9       2
4-10      2
12-0      2
Name: count, Length: 95, dtype: int64


In [61]:
print(WAB_df.dtypes)

print("Null Counts:")
for columns in WAB_df:
    print(WAB_df.isnull().sum())

print("Rows per season:")
print(WAB_df["season"].value_counts())

print(WAB_df.describe())

season             int64
team                 str
wab              float64
actual_wins        int64
expected_wins    float64
games_played       int64
elo                int64
elo_rank           int64
dtype: object
Null Counts:
season           0
team             0
wab              0
actual_wins      0
expected_wins    0
games_played     0
elo              0
elo_rank         0
dtype: int64
season           0
team             0
wab              0
actual_wins      0
expected_wins    0
games_played     0
elo              0
elo_rank         0
dtype: int64
season           0
team             0
wab              0
actual_wins      0
expected_wins    0
games_played     0
elo              0
elo_rank         0
dtype: int64
season           0
team             0
wab              0
actual_wins      0
expected_wins    0
games_played     0
elo              0
elo_rank         0
dtype: int64
season           0
team             0
wab              0
actual_wins      0
expected_wins    0
games_played     0

Clean DATA

In [51]:
NCCA_Data = NCAA_Data.drop_duplicates()
print(NCAA_Data)

          year              Team     Conference  NET Rank  PrevNET  \
0     Combined           Gonzaga            WCC         1        1   
1     Combined            Kansas         Big 12         2        2   
2     Combined            Dayton    Atlantic 10         3        3   
3     Combined     San Diego St.  Mountain West         4        4   
4     Combined            Baylor         Big 12         5        5   
...        ...               ...            ...       ...      ...   
5019      2020            Howard           MEAC       349      349   
5020      2020              UMES           MEAC       350      350   
5021      2020  Mississippi Val.           SWAC       351      351   
5022      2020      Kennesaw St.           ASUN       352      352   
5023      2020       Chicago St.            WAC       353      353   

      AvgOppNETRank  AvgOppNET    WL Conf.Record Non-ConferenceRecord RoadWL  \
0               108        146  31-2        17-1                 14-1   10-1   